# Notebook 07 â€” Integration Test: World Model Pipeline

Mocks the full ARGUS pipeline in Colab â€” no cameras needed. Verifies all models work together before Jetson deployment.

## Cell 07-01 | Mount & Load All Models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch, cv2, time
import numpy as np

BASE    = '/content/drive/MyDrive/ARGUS'
MODELS  = f'{BASE}/models'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Object Detection (YOLOv8)
from ultralytics import YOLO
det_path = f'{MODELS}/detection/yolov8s_argus_final.pt'
detector = YOLO(det_path) if os.path.exists(det_path) else YOLO('yolov8s.pt')
print('Object detector loaded')

# 2. Segmentation (SegFormer)
from transformers import SegformerForSemanticSegmentation
seg_path = f'{MODELS}/segmentation/segformer_b2_argus'
if os.path.exists(seg_path):
    segmentor = SegformerForSemanticSegmentation.from_pretrained(seg_path).to(DEVICE).eval()
    print('Segmentation model loaded')
else:
    segmentor = None
    print('Segmentation model not yet trained')

# 3. LLM
from llama_cpp import Llama
llm_path = f'{MODELS}/llm/Phi-3.5-mini-instruct-Q4_K_M.gguf'
llm = Llama(model_path=llm_path, n_ctx=2048, n_gpu_layers=35, verbose=False) if os.path.exists(llm_path) else None
print('LLM loaded' if llm else 'LLM not yet downloaded')

# 4. Face Privacy Filter
try:
    from insightface.app import FaceAnalysis
    face_app = FaceAnalysis(root=f'{MODELS}/privacy',
                            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    face_app.prepare(ctx_id=0, det_size=(640, 640))
    print('Face privacy filter loaded')
except:
    face_app = None
    print('Face filter not loaded')

# 5. STT
from faster_whisper import WhisperModel
stt = WhisperModel('tiny', device='cpu', compute_type='int8', download_root=f'{MODELS}/whisper')
print('Whisper STT loaded')

print()
print('=' * 50)
print('ARGUS Pipeline Ready')

## Cell 07-02 | World Model Builder Class

In [ ]:
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T

ARGUS_CLASSES = {
    0: 'background', 1: 'navigable_floor', 2: 'wall',
    3: 'door_closed', 4: 'door_open', 5: 'stairs_up',
    6: 'stairs_down', 7: 'person', 8: 'furniture', 9: 'clutter'
}

SEG_TRANSFORM = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class WorldModelBuilder:
    # Fuses detector + segmentor + privacy filter into a single World Model dict
    def __init__(self, detector, segmentor, face_app, device):
        self.detector  = detector
        self.segmentor = segmentor
        self.face_app  = face_app
        self.device    = device

    def process_frame(self, wide_frame_bgr, depth_map=None):
        # Process one BGR frame and return World Model dict
        h, w = wide_frame_bgr.shape[:2]
        world = {'timestamp': time.time(), 'objects': [], 'navigable_floor': True,
                 'nearest_obstacle_dist': 999.0, 'hazard': None, 'segmentation_summary': ''}

        # Object Detection
        results = self.detector(wide_frame_bgr, verbose=False)[0]
        for box in results.boxes:
            label = results.names[int(box.cls)]
            conf  = float(box.conf)
            if conf < 0.35: continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx  = (x1 + x2) / 2 / w
            dir = 'left' if cx < 0.35 else 'right' if cx > 0.65 else 'center'
            if depth_map is not None:
                cy_px = int((y1+y2)/2); cx_px = int((x1+x2)/2)
                dist  = float(np.median(depth_map[max(0,cy_px-10):cy_px+10, max(0,cx_px-10):cx_px+10]))
            else:
                dist = max(0.3, 1.0 / ((y2-y1)/h + 0.01))
            world['objects'].append({'label': label, 'distance': round(dist,2), 'direction': dir, 'private': False, 'conf': round(conf,2)})

        # Privacy: Face Detection
        if self.face_app:
            for face in self.face_app.get(cv2.cvtColor(wide_frame_bgr, cv2.COLOR_BGR2RGB)):
                world['objects'].append({'label': 'person_face', 'distance': 1.5, 'direction': 'center', 'private': True})

        # Segmentation -> Hazard Detection
        if self.segmentor:
            pil    = Image.fromarray(cv2.cvtColor(wide_frame_bgr, cv2.COLOR_BGR2RGB))
            tensor = SEG_TRANSFORM(pil).unsqueeze(0).to(self.device)
            with torch.no_grad():
                logits   = F.interpolate(self.segmentor(pixel_values=tensor).logits, size=(h,w), mode='bilinear', align_corners=False)
            seg_mask = logits.argmax(dim=1).squeeze().cpu().numpy()
            if (seg_mask == 6).sum() > 0.02 * h * w: world['hazard'] = 'stairs_down'
            elif (seg_mask == 5).sum() > 0.02 * h * w: world['hazard'] = 'stairs_up'
            floor_area = (seg_mask == 1).sum() / (h * w)
            world['navigable_floor']      = floor_area > 0.1
            world['segmentation_summary'] = f'floor={floor_area:.0%}, hazard={world["hazard"] or "none"}'

        # Nearest obstacle
        visible = [o for o in world['objects'] if not o.get('private')]
        if visible:
            world['nearest_obstacle_dist'] = min(o['distance'] for o in visible)
        return world

builder = WorldModelBuilder(detector, segmentor, face_app, DEVICE)
print('WorldModelBuilder ready')

## Cell 07-03 | Full Pipeline Integration Test

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


import cv2, numpy as np
import matplotlib.pyplot as plt

test_path = '/tmp/test_indoor.jpg'
try:
    TEST_URL = 'https://images.unsplash.com/photo-1556909114-f6e7ad7d3136?w=640&q=80'
    _safe_dl(TEST_URL, test_path)  # resume if partial
except Exception as e:
    print(f'Test image download failed ({e}) — using synthetic image')
    import numpy as np; import cv2
    synthetic = np.random.randint(80, 180, (480, 640, 3), dtype=np.uint8)
    cv2.imwrite(test_path, synthetic)

img = cv2.imread(test_path)
assert img is not None, f'Could not read {test_path}'
print(f'Test image loaded: {img.shape}')

## Cell 07-04 | Performance Benchmark

In [ ]:
import time, numpy as np

print('ARGUS Component Latency Benchmark (Colab T4 GPU)')
print('=' * 60)
frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# Detection benchmark
times = [(lambda t: (detector(frame, verbose=False), time.time()-t))(time.time())[1]*1000 for _ in range(10)]
det_ms = float(np.mean(times[2:]))

results = {'Detection (YOLOv8)': (det_ms, 50, '20-50ms')}

if segmentor:
    pil    = Image.fromarray(frame)
    tensor = SEG_TRANSFORM(pil).unsqueeze(0).to(DEVICE)
    times  = []
    for _ in range(5):
        t = time.time()
        with torch.no_grad(): segmentor(pixel_values=tensor)
        times.append((time.time()-t)*1000)
    results['Segmentation (SegFormer)'] = (float(np.mean(times[1:])), 100, '50-100ms')

print(f'{"Component":<30} {"Latency":>10} {"Target":>10} {"Status":>8}')
print('-' * 60)
for name, (lat, tgt_ms, tgt_str) in results.items():
    status = 'PASS' if lat <= tgt_ms else 'SLOW'
    print(f'{name:<30} {lat:>8.0f}ms {tgt_str:>10} {status:>8}')
print('-' * 60)
print('Jetson Orin Nano Super: ~1.5-2x slower than Colab T4.')
print('INT8 TensorRT on Jetson brings all models into target range.')

## Cell 07-05 | Save Models Manifest

In [ ]:
import json, os

manifest = {
    'argus_models': {
        'stereo_depth'   : {'pytorch': f'{MODELS}/depth/raft_stereo_final.pth',
                            'onnx': f'{EXPORTS}/tensorrt/raft_stereo_640x480.onnx',
                            'input': '2x [1,3,480,640]', 'fps_jetson': '15-20'},
        'segmentation'   : {'pytorch': f'{MODELS}/segmentation/segformer_b2_argus',
                            'onnx': f'{EXPORTS}/tensorrt/segformer_b2_512x512.onnx',
                            'input': '[1,3,512,512] normalized', 'fps_jetson': '10-15',
                            'classes': list(ARGUS_CLASSES.values())},
        'detection'      : {'pytorch': f'{MODELS}/detection/yolov8s_argus_final.pt',
                            'onnx': f'{EXPORTS}/tensorrt/yolov8s_argus_640.onnx',
                            'fps_jetson': '20-30'},
        'privacy_face'   : {'type': 'insightface SCRFD', 'weights': f'{MODELS}/privacy/'},
        'speech_wakeword': {'model': f'{MODELS}/wakeword_argus/argus.tflite', 'type': 'openWakeWord TFLite'},
        'speech_stt'     : {'model': f'{MODELS}/whisper', 'type': 'faster-whisper tiny INT8'},
        'speech_tts'     : {'model': f'{MODELS}/piper/en_US-lessac-medium.onnx', 'type': 'Piper TTS ONNX'},
        'llm'            : {'model': f'{MODELS}/llm/Phi-3.5-mini-instruct-Q4_K_M.gguf',
                            'type': 'llama.cpp GGUF INT4', 'context': 2048, 'latency': '<400ms'},
    },
    'jetson_deployment': {
        'trt_command'        : 'trtexec --onnx=<model>.onnx --fp16 --saveEngine=<model>.trt',
        'pipeline_entrypoint': 'argus_pipeline.py',
    }
}

MANIFEST_PATH = f'{BASE}/models/models_manifest.json'
with open(MANIFEST_PATH, 'w') as f: json.dump(manifest, f, indent=2)
print(f'Models manifest saved: {MANIFEST_PATH}')